In [1]:
import os
from pathlib import Path
import scipy.io as sio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
import re


In [ ]:
# Folder containing participant .mat logfiles
data_folder = r"D:\Athulya_Desktop_copied\Athulya_Krishnan_dc\Athulya_Krishnan\Athulya\IISc\Sagarika_IEL\IEL_partDat"

# Participant ID
# Participant ID loop (e.g., 1 to 10 or more)
for participant in range(1,39):

    # Standardized ID formatting (e.g., IEL001, IEL011)
    if participant < 10:
        part_id = f"IEL00{participant}"
    else:
        part_id = f"IEL0{participant}"

    # Find files matching 'IEL001*.mat'
    mat_files = glob(os.path.join(data_folder, f"{part_id}*.mat"))
    def extract_last_number(filename):
        numbers = re.findall(r'(\d+)(?=\.mat)', filename)
        if numbers:
            return int(numbers[-1])
        else:
            return 0
        
    all_runs = []

    # Excluding participant
    if part_id == "IEL006":
        print(f"--> Skipping completely removed participant: {part_id}")
        continue
    

    for file in mat_files:
        mat = sio.loadmat(file)
        run_number = extract_last_number(file)


        if part_id == "IEL010" and run_number == 3:
            print(f"!!! Explicitly excluding Run {run_number} for {part_id} !!!")
            continue
        
        if part_id == "IEL033" and run_number == 4:
            print(f"!!! Explicitly excluding Run {run_number} for {part_id} !!!")
            continue

        # Extract and forcefully flatten/squeeze variables

        # np.squeeze removes redundant 1D dimensions (e.g., shape (1, 36) -> (36,))
        trial_no   = np.squeeze(mat['trialIndex']).ravel()
        loss       = np.squeeze(mat['LossList']).ravel()
        emotion    = np.squeeze(mat['EmoList']).ravel()
        rt         = np.squeeze(mat['RT']).ravel()
        accuracy   = np.squeeze(mat['accuracy']).ravel()
        cue_onset  = np.squeeze(mat['CueOnset']).ravel()
        task_onset = np.squeeze(mat['TaskOnset']).ravel()
        isi        = np.squeeze(mat['isivalues']).ravel()
        iti        = np.squeeze(mat['itivalues']).ravel()

     
        # Loss condition labels
        loss_labels = []

        for val in loss:
            if val == 1:
                loss_labels.append("Loss")
            elif val == 2:
                loss_labels.append("NoLoss")
            else:
                loss_labels.append("Unknown")

        # Emotion labels
        emotion_labels = []

        for val in emotion:
            if val == 1:
                emotion_labels.append("Positive")
            elif val == 2:
                emotion_labels.append("Neutral")
            else:
                emotion_labels.append("Unknown")

        
        # Create dataframe
        df = pd.DataFrame({
            'Participant' : part_id,       # Saved as 'IEL001' instead of just 1
            'Run'         : run_number,
            'TrialNumber' : trial_no,
            'Loss'        : loss_labels,
            'Emotion'     : emotion_labels,
            'RT'          : rt,
            'Accuracy'    : accuracy,
            'CueOnset'    : cue_onset - 5,
            'TaskOnset'   : task_onset - 5,
            'ISI'         : isi,
            'ITI'         : iti,})

        all_runs.append(df)
    
    if not all_runs:
        print(f"No files found for participant {part_id}")
        continue

    # Concatenate and Clean
    final_df = pd.concat(all_runs, ignore_index=True)
    
    # Cast to ensure sorting works perfectly
    final_df['Run'] = pd.to_numeric(final_df['Run'], errors='coerce')
    final_df['TrialNumber'] = pd.to_numeric(final_df['TrialNumber'], errors='coerce')
    
    # filtering the correct trials only
    final_clean_df = final_df[(final_df['Accuracy'] != 0) & (final_df['Accuracy'].notna())].copy()
    print(final_clean_df.shape)

    #groupby condition
    group_condition = final_clean_df.groupby(['Emotion', 'Loss'])['RT']
    mean_condition = group_condition.transform('mean')
    std_condition = group_condition.transform('std')

    # limits for outlier detection
    upper_limit = mean_condition + (3 * std_condition)
    lower_limit = mean_condition - (3 * std_condition)

    # final filtering based on outlier removal
    outlier_df = (final_clean_df['RT'] > lower_limit) | (final_clean_df['RT'] < upper_limit)
    final_condition_df = final_clean_df[outlier_df]
    print(final_condition_df.shape)

    # Sort chronologically by Run then Trial
    final_condition_df = final_condition_df.sort_values(by=['Run', 'TrialNumber'])

    # Drop explicit duplicate entries if MATLAB logged the same trial twice
    #final_condition_df = final_condition_df.drop_duplicates(subset=['Run', 'TrialNumber'], keep='first')

    # Visualisation
    plt.figure()
    sns.barplot(final_condition_df, x = 'Emotion', y='RT', hue = 'Loss', errorbar='ci')
    plt.title('Behaviour RT')
    plt.xlabel('Emotion')
    plt.ylabel('RT(s)')
    plt.show()

    print(f"--- Summary for {part_id} ---")
    print("Total Unique Rows:", len(final_condition_df))
    print(final_condition_df)

    plt.figure()
    sns.barplot(final_condition_df, x= 'Emotion', y='Accuracy', hue='Loss', errorbar = 'ci')
    plt.title('Accuracy (%)')
    plt.xlabel('Emotion')
    plt.ylabel('Acc(%)')
    plt.show()

    # Save CSV file
    # output_name = f"{part_id}_master_file.csv"
    # final_condition_df.to_csv(output_name, index=False)
    # print(f"Saved: {output_name}\n")
